# Evo-1: Proprioceptive State Utilization Analysis

## Research Question
**Hypothesis**: Proprioceptive state information is NOT being fully utilized by the state encoder.

## Experimental Design
1. **Pass normal state** → Measure gradient flow to action_head.state_encoder
2. **Pass zeros (ablation)** → Measure gradient flow to action_head.state_encoder
3. **Compare**: If gradients barely change, state is underutilized

## What We'll Measure
- Gradient magnitude at state_encoder (CategorySpecificMLP inside action_head)
- Baseline contribution ratio (state encoder / total model gradients)
- Gradient change percentage when state encoder is ablated
- Retention/reduction ratios to understand state encoder importance

**Model**: Evo-1 (0.77B parameters)  
**Architecture** (VERIFIED from MINT-SJTU/Evo-1):  
- embedder: InternVL3-1B (VL backbone)
- action_head: FlowmatchingActionHead
  - **state_encoder**: CategorySpecificMLP (3-layer MLP: state → embeddings)
  
**Key Metric**: If zero state → minimal gradient change = underutilization

---

# Setup & Analysis

Simply run all cells below to complete the full analysis.

## 1. Environment Check

In [ ]:
# Check environment
import sys
IN_COLAB = 'google.colab' in sys.modules

print("🚀 Running on Google Colab")
!nvidia-smi

## 2. Environment Setup for Evo-1

This will create 3 conda environments:
1. **evo1_server** - For Evo-1 model (Python 3.10, PyTorch 2.5.1, transformers==4.57.6, Flash Attention)
2. **libero_client** - For LIBERO benchmark (Python 3.8.13)
3. **metaworld_client** - For MetaWorld benchmark (Python 3.10)

**Note**: Flash Attention installation takes 10-15 minutes (compiles from source).

In [ ]:
# Install dependencies (3 SEPARATE conda environments - official structure)
import os

print('📦 Installing system dependencies...')
!apt-get update -qq
!apt-get install -y -qq git wget ffmpeg libosmesa6-dev patchelf libgl1-mesa-glx libegl1-mesa > /dev/null 2>&1

print('📦 Installing Miniconda...')
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/conda > /dev/null 2>&1
!rm /tmp/miniconda.sh
os.environ['PATH'] = f"/opt/conda/bin:{os.environ['PATH']}"
!conda init bash
!conda config --set always_yes yes

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

print('\n📦 Creating 3 conda environments (official Evo-1 structure)...')
print('='*60)

# Environment 1: Evo-1 server (Python 3.10)
print('\n[1/3] evo1_server (Python 3.10) - for Evo-1 model')
!conda create -n evo1_server python=3.10 -y
!conda run -n evo1_server pip install \
  torch==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

!conda run -n evo1_server pip install \
  'numpy>=1.26.4,<2.0' 'transformers==4.57.6' accelerate diffusers \
  einops timm pillow opencv-python-headless \
  websockets pyyaml huggingface_hub tqdm

print('📦 Installing flash-attn (required, ~10-15 min)...')
!conda run -n evo1_server pip install flash-attn --no-build-isolation 2>&1 | tail -5 || echo '⚠️ Flash-attn failed'
print('✅ evo1_server ready')

# Environment 2: LIBERO client (Python 3.8.13 - OFFICIAL requirement)
print('\n[2/3] libero_client (Python 3.8.13) - for LIBERO benchmark')
!conda create -n libero_client python=3.8.13 -y
!conda run -n libero_client pip install \
  'numpy>=1.20,<2.0' robosuite==1.4.1 mujoco==2.3.7 \
  imageio h5py bddl websockets huggingface_hub
!conda run -n libero_client pip install \
  torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 \
  --extra-index-url https://download.pytorch.org/whl/cu113
print('✅ libero_client ready')

# Environment 3: MetaWorld client (Python 3.10)
print('\n[3/3] metaworld_client (Python 3.10) - for MetaWorld benchmark')
!conda create -n metaworld_client python=3.10 -y
!conda run -n metaworld_client pip install \
  mujoco websockets opencv-python packaging huggingface_hub metaworld gymnasium
print('✅ metaworld_client ready')

print('\n' + '='*60)
print('✅ All 3 environments created!')
!conda run -n evo1_server python -c "import sys; print(f'  evo1_server: Python {sys.version.split()[0]}')"
!conda run -n libero_client python -c "import sys; print(f'  libero_client: Python {sys.version.split()[0]}')"
!conda run -n metaworld_client python -c "import sys; print(f'  metaworld_client: Python {sys.version.split()[0]}')"

print('\n✅ Setup complete!')
print('='*60)

## 3. Clone Repositories

In [ ]:
# Clone Evo-1 repository
import os
from pathlib import Path

print('📦 Cloning Evo-1 repository...')
evo1_repo_path = '/content/Evo-1'

if not Path(f'{evo1_repo_path}/.git').exists():
    !git clone https://github.com/MINT-SJTU/Evo-1.git {evo1_repo_path}
    print('✅ Evo-1 cloned')
else:
    print('✅ Evo-1 already exists')

In [ ]:
# Clone EmbodiedLLM repository (contains analysis scripts and hooks)
!git clone https://github.com/PrithviTM-glitch/EmbodiedLLM.git
%cd EmbodiedLLM
!git checkout AnalyseMultipleHooks
%cd MultipleHooksStudy
print("✅ Analysis framework ready")

## 4. Download Checkpoints

In [ ]:
# Download Evo-1 checkpoints
from huggingface_hub import snapshot_download
from pathlib import Path

CHECKPOINTS_DIR = Path('/content/checkpoints/')
CHECKPOINTS = {
    'libero': {'repo': 'MINT-SJTU/Evo1_LIBERO', 'dir': CHECKPOINTS_DIR / 'libero'},
    'metaworld': {'repo': 'MINT-SJTU/Evo1_MetaWorld', 'dir': CHECKPOINTS_DIR / 'metaworld'}
}

print('📥 Downloading Evo-1 checkpoints...')
print('='*60)

for name, cfg in CHECKPOINTS.items():
    cfg['dir'].mkdir(parents=True, exist_ok=True)
    model_file = cfg['dir'] / 'mp_rank_00_model_states.pt'
    
    if model_file.exists() and model_file.stat().st_size > 1_000_000:
        print(f"✅ {name}: {model_file.stat().st_size / 1e9:.2f} GB (cached)")
    else:
        print(f"⏳ Downloading {name} from {cfg['repo']}...")
        snapshot_download(
            repo_id=cfg['repo'],
            local_dir=str(cfg['dir']),
            local_dir_use_symlinks=False,
        )
        if model_file.exists():
            print(f"✅ {name}: {model_file.stat().st_size / 1e9:.2f} GB")

print('\n✅ All checkpoints ready')
print('='*60)

## 5. Run Gradient Flow Analysis

This automated script will:
1. Load Evo-1 model in `evo1_server` conda environment
2. Attach gradient hooks
3. Run baseline analysis with normal state encoder
4. Run ablation analysis with zeroed state encoder output
5. Compare gradients and generate metrics

**Metrics Computed:**

*Baseline Metrics (no ablation):*
- **Baseline Contribution Ratio**: state_encoder_grad / total_model_grad  
  Shows what fraction of total model gradients flow through the state encoder

*Ablation-Based Metrics (comparing baseline vs ablated):*
- **Gradient Change %**: Absolute difference as percentage of baseline
- **Retention Ratio**: Fraction remaining after ablation (e.g., 0.60 = 60% retained)
- **Reduction Ratio**: Fraction lost (e.g., 0.40 = state encoder contributes 40%)

**Interpretation:**
- High baseline contribution (>0.05) + High reduction ratio (>0.3) → State encoder is critical
- Low baseline contribution (<0.02) + Low reduction ratio (<0.1) → Vision/language-dominated

In [ ]:
# Run complete gradient flow analysis
import os

print('🔬 Running Evo-1 Gradient Flow Analysis')
print('='*60)
print('This will:')
print('  1. Load Evo-1 model (in evo1_server conda environment)')
print('  2. Attach gradient hooks')
print('  3. Run baseline analysis (normal state)')
print('  4. Run ablation analysis (zeroed state encoder)')
print('  5. Compare gradients and generate verdict')
print('='*60)
print()

os.chdir('/content/EmbodiedLLM/MultipleHooksStudy')

print('⏳ Running analysis (takes 2-3 minutes)...')
print()

!conda run -n evo1_server python scripts/run_evo1_gradient_analysis.py --checkpoint metaworld

print('\n✅ Analysis complete!')
print('   Results saved to: /content/evo1_gradient_analysis_results.json')
print('   Run next cell to display results')

## 6. Display Results

In [ ]:
# Display the results from the gradient analysis
import json
import os

results_file = '/content/evo1_gradient_analysis_results.json'

if not os.path.exists(results_file):
    print('❌ Results file not found!')
    print('   Please run the previous cell first')
else:
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    print('\n' + '='*60)
    print('EVO-1 GRADIENT FLOW ANALYSIS RESULTS')
    print('='*60)
    
    print(f"\n📊 Model Information:")
    print(f"   Model: {results['model']}")
    print(f"   Checkpoint: {results['checkpoint']}")
    print(f"   Device: {results['device']}")
    print(f"   State Encoder: {results['state_encoder']}")
    
    print(f"\n🔬 Gradient Analysis:")
    print(f"   Ablation Method: {results['ablation_method']}")
    print(f"   \n   BASELINE (Normal State):")
    print(f"   Baseline Gradient Norm:  {results['baseline_grad_norm']:.6f}")
    print(f"   Total Model Grad Norm:   {results['total_model_grad_norm']:.6f}")
    print(f"   Baseline Contribution:   {results['baseline_contribution_ratio']:.6f} ({results['baseline_contribution_ratio']*100:.2f}% of total)")
    print(f"   \n   ABLATION (Zeroed State Encoder):")
    print(f"   Ablated Gradient Norm:   {results['ablated_grad_norm']:.6f}")
    print(f"   Absolute Reduction:      {results['gradient_absolute_reduction']:.6f}")
    print(f"   Gradient Change:         {results['gradient_change_pct']:.1f}%")
    print(f"   Retention Ratio:         {results['gradient_retention_ratio']:.3f} ({results['gradient_retention_ratio']*100:.1f}% retained)")
    print(f"   Reduction Ratio:         {results['gradient_reduction_ratio']:.3f} ({results['gradient_reduction_ratio']*100:.1f}% lost)")
    
    print(f"\n📈 Loss Comparison:")
    print(f"   Baseline Loss:  {results['loss_baseline']:.6f}")
    print(f"   Ablated Loss:   {results['loss_ablated']:.6f}")
    
    print(f"\n🎯 VERDICT: {results['verdict']}")
    print(f"\n{results['explanation']}")
    
    print('\n' + '='*60)
    print('Methodology: Output ablation (zeroed state encoder output)')
    print('='*60)
    
    print(f"\n✅ Full results saved to: {results_file}")

## Advanced Options

To analyze the LIBERO checkpoint instead:
```python
!conda run -n evo1_server python /content/EmbodiedLLM/MultipleHooksStudy/scripts/run_evo1_gradient_analysis.py --checkpoint libero
```

To specify custom output location:
```python
!conda run -n evo1_server python /content/EmbodiedLLM/MultipleHooksStudy/scripts/run_evo1_gradient_analysis.py --checkpoint metaworld --output /content/my_results.json
```